# SIM Intervention Demand Forecasting — Chronological Validation

## 4.1. Objective

The objective of this stage is to establish a chronological validation framework for forecasting daily SIM intervention workloads across six operational divisions.

Unlike random data splitting, chronological validation preserves the order of observations and evaluates forecasts using historical information that would have been available at the time of prediction.

The validation framework will follow these project requirements:

- **Forecast target:** Daily intervention count for each operational division.
- **Forecast horizon:** One day ahead (D).
- **Forecast preparation:** Day D−1.
- **Historical data cutoff:** Intervention counts available through D−3.
- **Development and validation period:** 2021–2024, using earlier observations when needed for historical features.
- **Reserved final test period:** January 1, 2025, through September 24, 2026.
- **Model updates:** Monthly, following the proposed modeling design.
- **Evaluation:** Mean Absolute Error (MAE) and prediction bias, compared with the selected same-weekday historical average baseline.

The reserved final test period will not be used for model selection or hyperparameter tuning.

## 4.2. Import Required Libraries

We will import pandas and NumPy to work with the feature dataset and prepare chronological validation periods.

Additional libraries will be introduced only when needed.

In [ ]:

import pandas as pd
import numpy as np


## 4.3. Load the Modeling Dataset

We will load the supervised modeling dataset produced during the Feature Engineering stage.

This dataset contains known intervention counts, calendar features, lag features, and rolling average features.

We will verify its dimensions, date range, and operational divisions before defining the chronological validation periods.

In [ ]:

# Define the dataset path
file_path = "../../data/interim/modelling/SIM_Features_Modeling.csv"

# Load the dataset and parse the date column
df_modeling = pd.read_csv(
    file_path,
    parse_dates=["Date"]
)

# Sort observations chronologically
df_modeling = df_modeling.sort_values(
    ["Date", "DIVISION"]
).reset_index(drop=True)

# Display basic information
print("Dataset dimensions:", df_modeling.shape)
print("Start date:", df_modeling["Date"].min().date())
print("End date:", df_modeling["Date"].max().date())
print("Divisions:", sorted(df_modeling["DIVISION"].unique()))

# Preview the dataset
display(df_modeling.head())


Dataset dimensions: (14748, 14)
Start date: 2020-01-01
End date: 2026-09-24
Divisions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


,Date,DIVISION,Intervention_Count,DayOfWeek,Month,DayOfMonth,IsWeekend,DayOfYear,Lag_3,Lag_7,Lag_14,Lag_28,Rolling_Mean_7,Rolling_Mean_14
0,2020-01-01,1,35.0,2,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01,2,67.0,2,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01,3,55.0,2,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-01,4,70.0,2,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-01,5,64.0,2,1,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN


## 4.4. Define Chronological Data Periods

We will separate the modeling dataset into three chronological periods according to the project design:

- **Historical context (2020):** Earlier observations available to support historical feature calculations and model development.
- **Development and validation (2021–2024):** Observations used to evaluate forecasting models and select the modeling approach.
- **Reserved final test (2025–September 24, 2026):** Observations held out from model selection and hyperparameter tuning.

This initial separation does not yet define individual training and validation windows. Those windows will be established in the next steps.

The final test period will remain reserved until the modeling approach has been selected.

In [ ]:

# Define chronological boundaries
development_start = pd.Timestamp("2021-01-01")
development_end = pd.Timestamp("2024-12-31")

test_start = pd.Timestamp("2025-01-01")
test_end = pd.Timestamp("2026-09-24")

# Separate historical context
df_history = df_modeling[
    df_modeling["Date"] < development_start
].copy()

# Separate development and validation data
df_development = df_modeling[
    (df_modeling["Date"] >= development_start) &
    (df_modeling["Date"] <= development_end)
].copy()

# Reserve final test data
df_test = df_modeling[
    (df_modeling["Date"] >= test_start) &
    (df_modeling["Date"] <= test_end)
].copy()

# Display the number of observations
print("Historical context:", len(df_history))
print("Development and validation:", len(df_development))
print("Reserved final test:", len(df_test))

# Verify the total number of observations
total_rows = (
    len(df_history) +
    len(df_development) +
    len(df_test)
)

print("\nOriginal dataset:", len(df_modeling))
print("Total after separation:", total_rows)
print("All observations accounted for:", total_rows == len(df_modeling))


Historical context: 2196
Development and validation: 8760
Reserved final test: 3792

Original dataset: 14748
Total after separation: 14748
All observations accounted for: True


**Methodological Observation**

The supervised modeling dataset was successfully separated into three non-overlapping chronological periods.

The historical context contains 2,196 observations from 2020. The development and validation period contains 8,760 observations from 2021 through 2024. The reserved final test period contains 3,792 observations from January 2025 through September 24, 2026.

All 14,748 observations were accounted for, with no records lost during the separation.

The final test period will remain excluded from model selection and hyperparameter tuning. Chronological validation within the development period will be designed to respect the D−3 information cutoff.

## 4.5. Define Monthly Validation Windows

We will define monthly validation windows covering January 2021 through December 2024.

For each validation month, the training period will begin with the earliest available historical observations and end three calendar days before the first forecast date of that month.

This approach simulates monthly model updates while respecting the D−3 availability restriction for historical intervention counts.

The model will remain fixed during each validation month, while daily predictive features may use newly available historical counts through D−3.

At this stage, we will define and inspect the validation schedule. Model fitting and prediction will be implemented in subsequent stages.

In [ ]:

# Create the beginning date of each validation month
validation_months = pd.date_range(
    start="2021-01-01",
    end="2024-12-01",
    freq="MS"
)

# Store the validation schedule
validation_windows = []

for month_start in validation_months:

    # Find the last day of the validation month
    month_end = month_start + pd.offsets.MonthEnd(0)

    # Apply the D-3 training cutoff
    training_cutoff = month_start - pd.Timedelta(days=3)

    # Store the monthly window
    validation_windows.append({
        "Validation_Start": month_start,
        "Validation_End": month_end,
        "Training_Cutoff": training_cutoff
    })

# Convert the schedule to a DataFrame
df_validation_windows = pd.DataFrame(validation_windows)

print("Number of validation months:", len(df_validation_windows))

print("\nFirst five validation windows:")
display(df_validation_windows.head())

print("\nLast five validation windows:")
display(df_validation_windows.tail())


Number of validation months: 48

First five validation windows:


,Validation_Start,Validation_End,Training_Cutoff
0,2021-01-01,2021-01-31,2020-12-29
1,2021-02-01,2021-02-28,2021-01-29
2,2021-03-01,2021-03-31,2021-02-26
3,2021-04-01,2021-04-30,2021-03-29
4,2021-05-01,2021-05-31,2021-04-28



Last five validation windows:


,Validation_Start,Validation_End,Training_Cutoff
43,2024-08-01,2024-08-31,2024-07-29
44,2024-09-01,2024-09-30,2024-08-29
45,2024-10-01,2024-10-31,2024-09-28
46,2024-11-01,2024-11-30,2024-10-29
47,2024-12-01,2024-12-31,2024-11-28


**Methodological Observation**

A total of 48 monthly validation windows were defined for January 2021 through December 2024.

For each validation month, the training cutoff was set three calendar days before the first forecast date. This respects the D−3 historical data availability constraint at the beginning of each monthly evaluation period.

The validation schedule supports an expanding training history with monthly model updates. During each validation month, the model will remain fixed, while daily predictive features may incorporate historical intervention counts as they become available under the D−3 constraint.

The schedule was created without using observations from the reserved final test period.

## 4.6. Verify Training and Validation Observations

We will calculate the number of training and validation observations in each monthly window.

Training observations will include historical dates up to the corresponding training cutoff. Validation observations will include the dates within the selected month.

This verification will confirm that training and validation periods do not overlap and that the number of available training observations increases over time.

We will also examine the final validation month, which contains the known missing intervention date of December 31, 2024.

In [ ]:

# Store the observation counts for each monthly window
window_counts = []

for _, window in df_validation_windows.iterrows():

    # Select training observations
    training_data = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Select validation observations
    validation_data = df_development[
        (df_development["Date"] >= window["Validation_Start"]) &
        (df_development["Date"] <= window["Validation_End"])
    ]

    # Store the counts and verify chronological separation
    window_counts.append({
        "Validation_Month": window["Validation_Start"].strftime("%Y-%m"),
        "Training_Rows": len(training_data),
        "Validation_Rows": len(validation_data),
        "No_Overlap": (
            training_data["Date"].max() <
            validation_data["Date"].min()
        )
    })

# Create a summary DataFrame
df_window_counts = pd.DataFrame(window_counts)

print("First five validation windows:")
display(df_window_counts.head())

print("\nLast five validation windows:")
display(df_window_counts.tail())

print("\nAll windows have no overlap:", df_window_counts["No_Overlap"].all())

print(
    "Total validation observations:",
    df_window_counts["Validation_Rows"].sum()
)


First five validation windows:


,Validation_Month,Training_Rows,Validation_Rows,No_Overlap
0,2021-01,2184,186,True
1,2021-02,2370,168,True
2,2021-03,2538,186,True
3,2021-04,2724,180,True
4,2021-05,2904,186,True



Last five validation windows:


,Validation_Month,Training_Rows,Validation_Rows,No_Overlap
43,2024-08,10032,186,True
44,2024-09,10218,180,True
45,2024-10,10398,186,True
46,2024-11,10584,180,True
47,2024-12,10764,180,True



All windows have no overlap: True
Total validation observations: 8760


**Methodological Observation**

The 48 monthly validation windows were verified for chronological separation and observation coverage.

Training observations increase over time, from 2,184 rows in the first window to 10,764 rows in the final window, following an expanding-window approach.

All validation windows are chronologically separated from their corresponding training periods. Together, they contain 8,760 validation observations, covering the complete 2021–2024 development and validation period.

December 2024 contains 180 evaluable observations rather than 186 because intervention counts are unavailable for December 31, 2024, across all six divisions.

## 4.7. Verify the D−3 Training Cutoff

We will verify that each monthly training dataset ends exactly three calendar days before the beginning of its validation month.

This check will confirm that no training target extends beyond the permitted historical information cutoff.

We will also verify that the reserved final test period is excluded from all monthly training and validation windows.

In [ ]:

# Store the cutoff verification results
cutoff_checks = []

for _, window in df_validation_windows.iterrows():

    # Select training data for the monthly window
    training_data = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Identify the latest training date
    latest_training_date = training_data["Date"].max()

    # Calculate the expected D-3 cutoff
    expected_cutoff = (
        window["Validation_Start"] -
        pd.Timedelta(days=3)
    )

    # Verify the cutoff and the final test boundary
    cutoff_checks.append({
        "Validation_Month": window["Validation_Start"].strftime("%Y-%m"),
        "Latest_Training_Date": latest_training_date,
        "Expected_Cutoff": expected_cutoff,
        "Cutoff_Correct": latest_training_date == expected_cutoff,
        "Test_Excluded": latest_training_date < test_start
    })

# Create a verification table
df_cutoff_checks = pd.DataFrame(cutoff_checks)

print("First five cutoff checks:")
display(df_cutoff_checks.head())

print("\nLast five cutoff checks:")
display(df_cutoff_checks.tail())

print("\nAll training cutoffs correct:", df_cutoff_checks["Cutoff_Correct"].all())
print("Final test excluded from training:", df_cutoff_checks["Test_Excluded"].all())

print(
    "Validation ends before final test:",
    df_validation_windows["Validation_End"].max() < test_start
)


First five cutoff checks:


,Validation_Month,Latest_Training_Date,Expected_Cutoff,Cutoff_Correct,Test_Excluded
0,2021-01,2020-12-29,2020-12-29,True,True
1,2021-02,2021-01-29,2021-01-29,True,True
2,2021-03,2021-02-26,2021-02-26,True,True
3,2021-04,2021-03-29,2021-03-29,True,True
4,2021-05,2021-04-28,2021-04-28,True,True



Last five cutoff checks:


,Validation_Month,Latest_Training_Date,Expected_Cutoff,Cutoff_Correct,Test_Excluded
43,2024-08,2024-07-29,2024-07-29,True,True
44,2024-09,2024-08-29,2024-08-29,True,True
45,2024-10,2024-09-28,2024-09-28,True,True
46,2024-11,2024-10-29,2024-10-29,True,True
47,2024-12,2024-11-28,2024-11-28,True,True



All training cutoffs correct: True
Final test excluded from training: True
Validation ends before final test: True


**Methodological Observation**

The D−3 training cutoff was verified across all 48 monthly validation windows.

For every window, the latest training observation matched the expected cutoff date, three calendar days before the beginning of the validation month.

All training and validation windows remained within the development period, and the reserved 2025–2026 final test period was excluded.

These checks confirm that the chronological validation schedule respects the intended historical availability boundaries. Model training procedures will need to follow the same restrictions to prevent data leakage.

## 4.8. Save the Chronological Validation Schedule

We will save the verified monthly validation schedule for use in subsequent model training and evaluation stages.

The schedule contains the validation month, validation start and end dates, training cutoff, and the number of training and validation observations.

Saving this schedule will allow the same chronological boundaries to be applied consistently when evaluating different forecasting models.

The reserved final test period is not included in the validation schedule.

In [ ]:

# Create a copy of the validated schedule
df_validation_schedule = df_validation_windows.copy()

# Add the training and validation observation counts
df_validation_schedule["Training_Rows"] = df_window_counts["Training_Rows"]
df_validation_schedule["Validation_Rows"] = df_window_counts["Validation_Rows"]

# Define the output file
output_file = "../../data/interim/modelling/SIM_Chronological_Validation_Schedule.csv"

# Save the schedule
df_validation_schedule.to_csv(output_file, index=False)

# Confirm the output
print("Validation schedule saved:", output_file)
print("Number of monthly windows:", len(df_validation_schedule))

# Preview the saved schedule
display(df_validation_schedule.head())


Validation schedule saved: ../../data/interim/modelling/SIM_Chronological_Validation_Schedule.csv
Number of monthly windows: 48


,Validation_Start,Validation_End,Training_Cutoff,Training_Rows,Validation_Rows
0,2021-01-01,2021-01-31,2020-12-29,2184,186
1,2021-02-01,2021-02-28,2021-01-29,2370,168
2,2021-03-01,2021-03-31,2021-02-26,2538,186
3,2021-04-01,2021-04-30,2021-03-29,2724,180
4,2021-05-01,2021-05-31,2021-04-28,2904,186


## 4.9. Stage Conclusion — Chronological Validation

This stage established a chronological validation framework for forecasting daily SIM intervention workloads across six operational divisions.

The supervised modeling dataset was separated into three periods:

- **Historical context (2020):** 2,196 observations.
- **Development and validation (2021–2024):** 8,760 observations.
- **Reserved final test (January 2025–September 24, 2026):** 3,792 observations.

A total of 48 monthly validation windows were created using an expanding training history. For each validation month, the training cutoff was set three calendar days before the first forecast date, respecting the D−3 historical information constraint.

The verification confirmed that training and validation dates do not overlap, all 8,760 development-period observations are covered by the validation schedule, and the reserved final test period is excluded.

The verified schedule was saved as `SIM_Chronological_Validation_Schedule.csv`.

**Conclusion:** The chronological validation schedule is ready to support model training and comparison. The monthly training cutoff and daily feature availability restrictions must also be enforced during model implementation to prevent data leakage. The final test period remains reserved and has not been evaluated.